# Phase 1 초기 CATE 학습 노트북

**Phase 1(순수 RCT)이 끝난 후 1회만 돌리는 노트북**입니다.

### 앱별 모델 분리
앱별로 모델이 다릅니다. `APP_NAME`을 바꾸면 다른 앱의 모델을 학습할 수 있어요.

Phase 1에서는 모든 유저에게 trigger를 100% 랜덤 배정했습니다.  
이 데이터(`data/{APP_NAME}/rct_data.csv`, 8000명)로 첫 CATE 모델을 만듭니다.

이후에는 `weekly_training.ipynb`에서 매주 재학습합니다.  
(매주 데이터에서 20% 랜덤 유저를 활용)

### 이 노트북이 하는 일
1. RCT 데이터 로드 (Phase 1에서 수집)
2. ATE 분석 — trigger별 평균 클릭률 비교
3. CATE 모델 학습 (S-Learner)
4. 결과 확인 — 유저별 best trigger
5. 모델 저장 → `models/{APP_NAME}/cate_model.pkl`

### 핵심 개념
- **ATE** (Average Treatment Effect): trigger 간 평균 효과 차이
- **CATE** (Conditional Average Treatment Effect): 유저별 효과 차이
- **S-Learner**: 하나의 모델에 trigger를 feature로 포함하여 학습하는 방법

---
## Step 1: RCT 데이터 로드

Phase 1에서 모은 데이터입니다.  
**모든 유저가 100% 랜덤으로** 4개 trigger + control 중 하나를 받았습니다.  
(Phase 2 이후의 weekly 데이터와 다르게, 여기는 모델 추천 유저가 없음)

In [ ]:
import pandas as pd
import numpy as np
import json
import pickle
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 앱 이름 (onboarding.ipynb의 APP_NAME과 동일)
APP_NAME = "ablog"

# 앱별 모델 저장 디렉토리 생성
os.makedirs(f'../models/{APP_NAME}', exist_ok=True)

# RCT 데이터 로드 (앱별 디렉토리)
rct = pd.read_csv(f'../data/{APP_NAME}/rct_data.csv')

# app_id 컬럼이 있으면 해당 앱만 필터링
if 'app_id' in rct.columns:
    rct = rct[rct['app_id'] == APP_NAME].reset_index(drop=True)

print(f'앱: {APP_NAME}')
print(f'총 유저 수: {len(rct)}')
print(f'전체 모달 클릭률: {rct["modal_clicked"].mean():.1%}')
print()
print('Trigger별 배정 수:')
print(rct['assigned_trigger'].value_counts())

---
## Step 2: ATE 분석 — Trigger 간 평균 효과 차이

먼저 "trigger별로 평균적으로 효과가 있는지"를 확인합니다.  
여기서 효과가 없으면 CATE를 해도 의미가 없어요.

Phase 1은 control 그룹이 있으므로 control 대비 ATE를 계산합니다.

In [2]:
TRIGGERS = ['price_appeal', 'social_proof', 'scarcity', 'novelty']

# Control 그룹 클릭률
control = rct[rct['assigned_trigger'] == 'control']
control_rate = control['modal_clicked'].mean()
print(f'Control 클릭률: {control_rate:.1%} (n={len(control)})')
print()

# 각 Trigger vs Control 비교
print(f'{"Trigger":15s} | {"클릭률":>8s} | {"ATE":>8s} | {"p-value":>10s} | 유의미?')
print('-' * 65)

for trigger in TRIGGERS:
    treat = rct[rct['assigned_trigger'] == trigger]
    treat_rate = treat['modal_clicked'].mean()
    ate = treat_rate - control_rate
    
    _, p_val = stats.ttest_ind(treat['modal_clicked'], control['modal_clicked'])
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'n.s.'
    
    print(f'{trigger:15s} | {treat_rate:>7.1%} | {ate:>+7.1%} | {p_val:>10.4f} | {sig}')

print()
print('* p<0.05, ** p<0.01, *** p<0.001, n.s. = 유의미하지 않음')

Control 클릭률: 15.5% (n=1558)

Trigger         |      클릭률 |      ATE |    p-value | 유의미?
-----------------------------------------------------------------
price_appeal    |   18.2% |   +2.7% |     0.0465 | *
social_proof    |   24.7% |   +9.2% |     0.0000 | ***
scarcity        |   22.3% |   +6.7% |     0.0000 | ***
novelty         |   21.5% |   +5.9% |     0.0000 | ***

* p<0.05, ** p<0.01, *** p<0.001, n.s. = 유의미하지 않음


---
## Step 3: CATE 모델 학습 (S-Learner)

**S-Learner 방식**: 하나의 모델에 trigger를 one-hot feature로 포함하여 학습합니다.

```
입력: [25개 user features, trigger_onehot(4)] → 모달 클릭 여부
```

T-Learner(trigger별 별도 모델)과 다르게, 모든 데이터를 하나의 모델이 공유합니다.  
소규모 데이터에서도 안정적이고, trigger 간 feature 효과를 공유할 수 있습니다.

Phase 1 데이터는 전부 랜덤 배정이므로, 8000명 전원을 사용합니다.  
(weekly_training에서는 20% 랜덤 유저만 사용하는 것과 다름)

In [ ]:
# Feature 정의
UA_FEATURES = [
    'trackinglink_count', 'DA_count', 'SA_count',
    'unique_channel_count', 'channel_entropy', 'last_touch_is_da',
    'latency', 'recency', 'recent_touch_pressure',
    'touch_per_latency_hour', 'last1h_touch_count', 'recent_24h_ratio',
    'click_ratio', 'impression_count', 'is_single_touch_install'
]

INAPP_FEATURES = [
    'product_viewed_count', 'user_signin', 'product_addedtocart',
    'deeplink_open', 'home_viewed', 'addtowishlist',
    'onboarding', 'user_signup', 'total_events', 'n_event_types'
]

CATE_FEATURES = UA_FEATURES + INAPP_FEATURES

# S-Learner: 하나의 모델, trigger를 one-hot feature로 포함
# Phase 1이라 전원 RCT 데이터 사용 (control 포함 — treatment trigger만 학습)
# control 유저는 제외 (서빙 시 4개 treatment trigger 중 선택하므로)
df_treat = rct[rct['assigned_trigger'].isin(TRIGGERS)].copy()

X_rows = []
y_rows = []
for _, row in df_treat.iterrows():
    features = row[CATE_FEATURES].values.astype(float)
    trigger = row['assigned_trigger']
    trigger_onehot = [1 if t == trigger else 0 for t in TRIGGERS]
    X_rows.append(np.concatenate([features, trigger_onehot]))
    y_rows.append(row['modal_clicked'])

X_cate = np.array(X_rows)
y_cate = np.array(y_rows)

print(f'S-Learner 학습 데이터: {X_cate.shape[0]}행 x {X_cate.shape[1]}열 (25 features + 4 trigger one-hot)')
print(f'  (control {len(rct) - len(df_treat)}명 제외, treatment trigger만 사용)')
print()

# 단일 모델 학습
cate_model = RandomForestClassifier(
    n_estimators=200, max_depth=8, min_samples_leaf=20, random_state=42
)
cate_model.fit(X_cate, y_cate)

# Trigger별 통계
print('Trigger별 통계 (Phase 1 — 전원 RCT 데이터):')
for trigger in TRIGGERS:
    mask = df_treat['assigned_trigger'] == trigger
    y_t = df_treat.loc[mask, 'modal_clicked'].values
    print(f'  {trigger:15s}: n={mask.sum()}, 클릭률={y_t.mean():.1%}')

print('\nS-Learner 모델 학습 완료!')

---
## Step 4: 결과 확인 — 유저별 best trigger

전체 유저에 대해 각 trigger의 클릭 확률을 예측하고, best trigger를 선택합니다.

In [ ]:
# 전체 유저에 대해 trigger별 클릭 확률 예측 (S-Learner)
X_base = rct[CATE_FEATURES].values

trigger_probs = {}
for trigger in TRIGGERS:  # control 제외
    trigger_onehot = np.array([1 if t == trigger else 0 for t in TRIGGERS])
    # 모든 유저에 같은 trigger one-hot 붙이기
    X_with_trigger = np.hstack([X_base, np.tile(trigger_onehot, (len(X_base), 1))])
    trigger_probs[trigger] = cate_model.predict_proba(X_with_trigger)[:, 1]

prob_df = pd.DataFrame(trigger_probs)

# 유저별 best trigger
rct['predicted_best'] = prob_df.idxmax(axis=1)

print('=== 유저별 Best Trigger 분포 ===')
print(rct['predicted_best'].value_counts())
print()

# 처음 5명 예시
print('=== 처음 5명의 trigger별 클릭 확률 ===')
sample = prob_df.head(5).round(3)
sample['best'] = prob_df.head(5).idxmax(axis=1)
print(sample)

### 어떤 유저가 어떤 trigger를 선호하는지 분석

각 trigger를 best로 가진 유저들의 평균 프로필을 비교합니다.  
차이가 있으면 CATE 모델이 잘 작동하는 것!

In [ ]:
key_features = ['recent_touch_pressure', 'channel_entropy', 
                'product_viewed_count', 'user_signin']

print('=== Trigger별 유저 프로필 (평균) ===')
print(f'{"":25s}', end='')
for trigger in TRIGGERS:
    print(f'{trigger:>15s}', end='')
print(f'{"전체":>10s}')
print('-' * 90)

for feat in key_features:
    print(f'{feat:25s}', end='')
    for trigger in TRIGGERS:
        mask = rct['predicted_best'] == trigger
        val = rct.loc[mask, feat].mean()
        print(f'{val:>15.2f}', end='')
    print(f'{rct[feat].mean():>10.2f}')

---
## Step 5: 모델 저장

이 파일을 서버에 올리면 **exploration → optimized 모드로 자동 전환**됩니다.  
이후에는 `weekly_training.ipynb`에서 매주 갱신합니다.

In [ ]:
# CATE 모델 저장 (S-Learner: 단일 모델)
cate_data = {
    'model': cate_model,
    'treatment_triggers': TRIGGERS,
    'feature_names': CATE_FEATURES,
}

with open(f'../models/{APP_NAME}/cate_model.pkl', 'wb') as f:
    pickle.dump(cate_data, f)

size = os.path.getsize(f'../models/{APP_NAME}/cate_model.pkl') / 1024 / 1024
print(f'../models/{APP_NAME}/cate_model.pkl 저장 완료! ({size:.1f} MB)')
print(f'  S-Learner: 단일 모델 (trigger를 one-hot feature로 포함)')
print(f'  학습 데이터: Phase 1 RCT treatment 유저 {len(df_treat)}명 사용')
print()
print('이 파일을 서버에 업로드하면:')
print('  - 서버가 cate_model.pkl을 감지')
print('  - exploration 모드 → optimized 모드로 자동 전환')
print('  - 80% 모델 추천 + 20% 랜덤 배정 시작')

---
## Step 6: 테스트 — 유저 1명 예측 (S-Learner)

서버에서 optimized 모드일 때의 trigger score 예측을 재현합니다.  
S-Learner: 각 trigger의 one-hot을 바꿔가며 하나의 모델로 4번 예측.

In [ ]:
# 유저 1명 선택
user = rct.iloc[0]
user_features = user[CATE_FEATURES].values.astype(float)

print(f'===== 유저 {user["user_id"]} — Trigger Score 예측 (S-Learner) =====')
print()

scores = {}
for trigger in TRIGGERS:
    trigger_onehot = [1 if t == trigger else 0 for t in TRIGGERS]
    x = np.concatenate([user_features, trigger_onehot]).reshape(1, -1)
    prob = cate_model.predict_proba(x)[0][1]
    scores[trigger] = round(prob, 4)
    print(f'  {trigger:15s}: 클릭 확률 {prob:.4f}')

best = max(scores, key=scores.get)
print(f'\n  → best_trigger: {best}')

# API 응답 형태
print(f'\n===== API 응답 (예시) =====')
response = {
    'user_id': user['user_id'],
    'best_trigger': best,
    'trigger_scores': scores,
}
print(json.dumps(response, indent=2))

---
## Step 7: 서버에 CATE 모델 업로드

서버에 cate_model.pkl을 업로드하면 exploration → optimized 모드로 전환됩니다.

**SERVER_URL을 실제 서버 주소로 바꿔주세요.**

In [ ]:
import requests

# 서버 주소
SERVER_URL = "https://airbridge-entry-api-prototype.onrender.com"  # 실서버
# SERVER_URL = "http://localhost:8000"  # 로컬 테스트

filepath = f"../models/{APP_NAME}/cate_model.pkl"
filename = "cate_model.pkl"

print(f"=== CATE 모델 업로드 ({APP_NAME}) ===")
print(f"서버: {SERVER_URL}")
print()

try:
    with open(filepath, "rb") as f:
        resp = requests.post(
            f"{SERVER_URL}/v1/models/{APP_NAME}/upload",
            files={"file": (filename, f)},
        )
    
    if resp.status_code == 200:
        result = resp.json()
        print(f"  [OK] {filename} → {result['mode']}")
        print(f"  {result['message']}")
    else:
        print(f"  [ERROR] {resp.status_code}: {resp.text}")

except requests.ConnectionError:
    print(f"  [ERROR] 서버에 연결할 수 없습니다: {SERVER_URL}")
    print(f"  → 서버를 먼저 실행하거나, SERVER_URL을 확인하세요")